# Paper figures — Colab
Editable controller for Figures 1–6 and S1–S6. Upload frozen Phase II–VI artifacts, customize palette/labels, and regenerate figures.

In [ ]:

from pathlib import Path
import io,re,zipfile
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
BASE=Path('/content'); OUT=BASE/'paper_figures'; (OUT/'main').mkdir(parents=True,exist_ok=True); (OUT/'supp').mkdir(exist_ok=True)
if not (BASE/'painting-geometry').exists():
 !git clone -q --depth 1 --branch multiscale-corpus-analysis https://github.com/ardominguezm/painting-geometry.git /content/painting-geometry


## Upload / paths

In [ ]:

# Upload the result ZIPs (or edit the paths below).
# uploaded=files.upload()
P={
 'p2':BASE/'phase2_inspect2.zip','p3':BASE/'painting_geometry_phase3_results.zip','p3b':BASE/'painting_geometry_phase3b_results.zip',
 'p4':BASE/'painting_geometry_phase4_artbench_pilot.zip','p4b':BASE/'painting_geometry_phase4b_scale_hierarchy.zip',
 'p5':BASE/'painting_geometry_phase5_style_geometry.zip','p6':BASE/'painting_geometry_phase6_ordinal_head_to_head.zip'}
REF=BASE/'the_starry_night.jpg'; DPI=300
C={'base':'#2f2f2f','gray':'#d9dde3','op':'#6b7d2b','k':'#1e4fa8','mix':'#8c406c','s1':'#245b73','s2':'#1e4fa8','s4':'#b8860b','s8':'#b85c38'}


In [ ]:

def get(z,n):
 z=Path(z)
 if z.is_dir(): return pd.read_csv(list(z.rglob(n))[0])
 with zipfile.ZipFile(z) as f:
  m=[x for x in f.namelist() if Path(x).name==n][0]; return pd.read_csv(io.BytesIO(f.read(m)))
def save(fig,name,supp=False):
 d=OUT/('supp' if supp else 'main')
 for e in ['png','pdf','svg']: fig.savefig(d/f'{name}.{e}',dpi=DPI if e=='png' else None,bbox_inches='tight')
 plt.close(fig)
def clean(a): a.spines[['top','right']].set_visible(False); a.grid(alpha=.25)


## Load results

In [ ]:

p2=get(P['p2'],'phase2_results.csv'); p2a=get(P['p2'],'phase2_per_artist_f1.csv'); p2p=get(P['p2'],'phase2_predictions.csv'); leaks=get(P['p2'],'leakage_threshold_sensitivity.csv')
p3=get(P['p3b'],'scale_ablation_results_clean.csv'); r3=get(P['p3'],'resolution_robustness_summary.csv'); starp=get(P['p3b'],'starry_covaware_pca_coordinates.csv')
p4=get(P['p4'],'artbench_artist_disjoint_results.csv'); p4b=get(P['p4b'],'phase4b_scale_hierarchy_results.csv'); p4bs=get(P['p4b'],'phase4b_scale_hierarchy_per_style.csv')
p5=get(P['p5'],'phase5_scale_summary.csv'); p6=get(P['p6'],'phase6_head_to_head_results.csv'); p6d=get(P['p6'],'phase6_head_to_head_deltas.csv')
print('Loaded')


## Figure builders
Edit any function, palette entry, title, labels, or layout, then rerun that figure.

In [ ]:

def f1():
 a=np.array(Image.open(REF).convert('RGB')); h,w,_=a.shape; y=.299*a[:,:,0]+.587*a[:,:,1]+.114*a[:,:,2]; fig,ax=plt.subplots(1,3,figsize=(12,4)); ax[0].imshow(a); ax[0].axis('off'); ax[1].imshow(y,cmap='gray'); ax[1].axis('off'); ax[2].contour(y,levels=np.quantile(y,[.2,.4,.6,.8])); ax[2].invert_yaxis(); ax[2].axis('off'); fig.suptitle('Figure 1. Multiscale level-set geometry pipeline'); save(fig,'Figure1_method_pipeline')
def f2():
 q=p2[p2.eval_set=='clean'].set_index('experiment').loc[['B_strong_full','G_geometry_full','BG_combined_full']]; fig,ax=plt.subplots(1,2,figsize=(11,4)); ax[0].bar(['B90','G44','B+G'],q.macro_f1,color=[C['base'],C['k'],C['mix']]); clean(ax[0]); z=p2a[(p2a.eval_set=='clean')&p2a.experiment.isin(['B_strong_full','BG_combined_full'])].pivot(index='artist',columns='experiment',values='f1'); ax[1].bar(z.index,z.BG_combined_full-z.B_strong_full,color=C['mix']); ax[1].tick_params(axis='x',rotation=45); clean(ax[1]); fig.suptitle('Figure 2. Artist-level complementarity'); save(fig,'Figure2_artist_complementarity')
def f3():
 q=p3[p3.experiment.isin(['S1','S2','S4','S8'])]; fig,ax=plt.subplots(figsize=(6,4)); ax.plot(['σ1','σ2','σ4','σ8'],q.macro_f1,'o-'); clean(ax); ax.set_ylabel('Macro-F1'); fig.suptitle('Figure 3. Scale-dependent discrimination'); save(fig,'Figure3_scale_tradeoff')
def f4():
 fig,ax=plt.subplots(1,2,figsize=(10,4));
 for j,ds in enumerate(['artbench10_all','artbench10_wikiart8']): q=p4[p4.dataset==ds].set_index('experiment').loc[['B_strong_full','G_geometry_full','BG_combined_full']]; ax[j].bar(['B90','G44','B+G'],q.macro_f1_oof,color=[C['base'],C['k'],C['mix']]); clean(ax[j]); ax[j].set_title(ds)
 fig.suptitle('Figure 4. Style generalization across unseen artists'); save(fig,'Figure4_style_generalization')
def f5():
 fig,ax=plt.subplots(1,2,figsize=(10,4)); order=['OP_HC','OP11','OP24','OP75','K40_curvature','OP75_K40']; labs=['HC','OP11','OP24','OP75','K40','OP+K']
 for j,ds in enumerate(['artbench10_all','artbench10_wikiart8']): q=p6[p6.dataset==ds].set_index('experiment').loc[order]; ax[j].bar(labs,q.macro_f1_oof,color=[C['op']]*4+[C['k'],C['mix']]); clean(ax[j]); ax[j].tick_params(axis='x',rotation=30); ax[j].set_title(ds)
 fig.suptitle('Figure 5. Ordinal patterns vs geometry'); save(fig,'Figure5_ordinal_vs_geometry')
def f6():
 fig,ax=plt.subplots(figsize=(6,4));
 for ds,c0 in [('artbench10_all',C['k']),('artbench10_wikiart8',C['mix'])]: q=p5[p5.dataset==ds]; ax.plot(q.sigma_ref,q.style_fraction,'o-',label=ds,color=c0)
 clean(ax); ax.legend(frameon=False); ax.set_xlabel('Scale'); ax.set_ylabel('Style fraction'); fig.suptitle('Figure 6. Style organization'); save(fig,'Figure6_style_organization')


In [ ]:

def s1():
 fig,ax=plt.subplots(figsize=(6,4));
 for m,c0 in [('B_strong_full',C['base']),('G_geometry_full',C['k']),('BG_combined_full',C['mix'])]: q=leaks[leaks.model==m]; ax.plot(q.threshold,q.macro_f1,'o-',label=m,color=c0)
 clean(ax); ax.legend(frameon=False); fig.suptitle('Figure S1. Leakage audit'); save(fig,'FigureS1_leakage_audit',True)
def s2():
 q=p2p[p2p.clean_eval==True]; arts=sorted(q.artist.unique()); fig,ax=plt.subplots(1,3,figsize=(12,4))
 for j,m in enumerate(['B_strong_full','G_geometry_full','BG_combined_full']): ct=pd.crosstab(q.artist,q[m],normalize='index').reindex(index=arts,columns=arts,fill_value=0); ax[j].imshow(ct,cmap='Blues',vmin=0,vmax=1); ax[j].set_title(m)
 fig.suptitle('Figure S2. Artist confusions'); save(fig,'FigureS2_artist_confusions',True)
def s3():
 fig,ax=plt.subplots(1,2,figsize=(10,4));
 for j,ds in enumerate(['artbench10_all','artbench10_wikiart8']): q=p4[p4.dataset==ds].set_index('experiment').loc[['B_strong_full','G_geometry_full','BG_combined_full','B_strong_k40','G_geometry_k40','BG_combined_k40']]; ax[j].bar(range(6),q.macro_f1_oof); clean(ax[j]); ax[j].set_title(ds)
 fig.suptitle('Figure S3. Full pilot benchmark'); save(fig,'FigureS3_phase4_pilot_details',True)
def s4():
 order=['G_s1','G_s2','G_s4','G_s8','G_fine_s12','G_coarse_s48','G_all_s1248']; fig,ax=plt.subplots(1,2,figsize=(10,4))
 for j,ds in enumerate(['artbench10_all','artbench10_wikiart8']): q=p4bs[(p4bs.dataset==ds)&p4bs.experiment.isin(order)].pivot(index='style',columns='experiment',values='f1_one_vs_rest'); ax[j].imshow(q,cmap='magma',aspect='auto'); ax[j].set_title(ds)
 fig.suptitle('Figure S4. Scale-by-style profiles'); save(fig,'FigureS4_scale_style_profiles',True)
def s5(): f6();
def s6(): f5();
ALL=[f1,f2,f3,f4,f5,f6,s1,s2,s3,s4,s5,s6]
def build_all():
 for fn in ALL: print('Building',fn.__name__); fn()
 print('Done:',OUT)


## Run
Examples: `f5()`, `s4()`, or `build_all()`. Change `C`, `DPI`, labels or any function above for future personalization.